In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from sklearn.neighbors import NearestNeighbors
import networkx as nx

# --- Metacell Aggregation Pipeline for Single-Cell Foundation Models (scGPT & Geneformer) ---
# Objective: Compress single-cell transcriptomes into high-fidelity metacells to mitigate
# technical dropouts and optimize the attention context window for transformer models.

# Assuming filtered target dataset with dimensions (5780 cells, 898 genes) is pre-loaded
# Replace 'adata_target' with your custom filtered variable if necessary.
adata_input = adata_target.copy()

print("Step 0: Validating and formatting input matrix layers...")
if not sp.issparse(adata_input.X):
    adata_input.X = sp.csr_matrix(adata_input.X)

# Convert to a dense numpy array for robust compatibility with Scikit-Learn downstream algorithms.
# A dimension of 898 highly-informative features fits comfortably into execution memory.
dense_matrix = adata_input.X.toarray()

print("\n Step 1: Performing K-Nearest Neighbors (KNN) search via Scikit-Learn...")
# Utilizing the 'ball_tree' algorithm ensures absolute CPU runtime stability and avoids kernel panics.
nn = NearestNeighbors(n_neighbors=15, metric='euclidean', algorithm='ball_tree', n_jobs=4)
nn.fit(dense_matrix)
distances, indices = nn.kneighbors(dense_matrix)

print("Step 1.2: Manually constructing the neighborhood adjacency graph...")
# Map the neighbor spatial coordinates into a NetworkX graph structure for community detection.
G = nx.Graph()
for i in range(dense_matrix.shape[0]):
    for j in indices[i]:
        if i != j:
            G.add_edge(i, j)

print("Step 1.5: Partitioning graph into compact micro-domains using Louvain modularity optimization...")
# Implement the community clustering algorithm to identify dense micro-scale cell networks.
try:
    import community as community_louvain
    # The 'resolution' parameter balances the size and density variance of the target metacells.
    partition = community_louvain.best_partition(G, resolution=2.0)

    # Assign community cluster metrics back to the single-cell observation matrix metadata.
    adata_input.obs['metacell_cluster'] = [str(partition[i]) if i in partition else "unclustered" for i in range(dense_matrix.shape[0])]
except ImportError:
    # Fail-safe alternative: Executed automatically if the 'python-louvain' module is not found.
    print("  [Warning] 'python-louvain' package not detected. Falling back to scalable MiniBatchKMeans...")
    from sklearn.cluster import MiniBatchKMeans
    kmeans = MiniBatchKMeans(n_clusters=250, random_state=42, batch_size=1024)
    adata_input.obs['metacell_cluster'] = kmeans.fit_predict(dense_matrix).astype(str)

print(f"Successfully derived distinct micro-clusters: {adata_input.obs['metacell_cluster'].nunique()}")

print("\nStep 2: Aggregating transcriptomic expressions and downstream metadata mapping...")
metacell_list = []
metacell_names = []
cluster_sizes_list = []
stages_list = []
patients_list = []
sample2_list = []

# ARCHITECTURAL FIX: Define continuous row indices to enable precise matrix slicing.
# This prevents index errors caused by duplicate cell barcodes or non-unique 'obs_names'.
adata_input.obs['row_idx'] = np.arange(adata_input.shape[0])

# Process each micro-cluster to build high-fidelity aggregated profiles
for cluster_id, group in adata_input.obs.groupby('metacell_cluster'):
    if cluster_id == "unclustered":
        continue

    # Isolate unique continuous row indices for the given cluster partition.
    cells_iloc = group['row_idx'].values
    cluster_size = len(cells_iloc)

    # Slicing is performed using structural integer locations (.X[cells_iloc, :]).
    # This vectorization method runs instantly and bypasses duplicate string index collisions.
    sub_matrix = adata_input.X[cells_iloc, :]

    # Calculate group-level arithmetic mean and flatten to a stable 1D expression vector.
    if sp.issparse(sub_matrix):
        mean_expression = np.array(sub_matrix.mean(axis=0)).ravel()
    else:
        mean_expression = np.array(np.mean(sub_matrix, axis=0)).ravel()

    metacell_list.append(mean_expression)
    metacell_names.append(f"metacell_{cluster_id}")
    cluster_sizes_list.append(cluster_size)

    # Extract clinical metadata using the modal value (most frequent observation) within the cluster.
    if 'stage' in group.columns:
        stages_list.append(group['stage'].mode()[0])
    elif 'clusters' in group.columns:
        stages_list.append(group['clusters'].mode()[0])
    else:
        stages_list.append("Unknown_Stage")

    if 'patient_id' in group.columns:
        patients_list.append(group['patient_id'].mode()[0])
    else:
        patients_list.append("Patient_Mock")

    if 'sample2' in group.columns:
        sample2_list.append(group['sample2'].mode()[0])
    else:
        sample2_list.append("Unknown_Response")


print("\nStep 3: Assembling the final compressed AnnData object structure...")
metacell_obs = pd.DataFrame(index=metacell_names)
metacell_obs['cell_count'] = cluster_sizes_list

# Append patient-level clinical characteristics and therapy responses
if stages_list: metacell_obs['stage'] = stages_list
if patients_list: metacell_obs['patient_id'] = patients_list
if sample2_list: metacell_obs['sample2'] = sample2_list

# Vertically stack all aggregated expression matrices into a compressed CSR matrix format
final_matrix = sp.csr_matrix(np.vstack(metacell_list))

# Inherit matching genomic properties from the initial variable index matrix
metacell_var = adata_input.var.copy()
metacell_var.index = metacell_var.index.astype(str)

# Instantiating the clean, aggregated metacell AnnData object
adata_metacells = sc.AnnData(
    X=final_matrix,
    obs=metacell_obs,
    var=metacell_var
)

print(f" Compression pipeline successfully executed! Final dimensions: {adata_metacells.shape}")

print(f"\n Summary Metrics:")
print(f"  Initial single-cell dataset size : {adata_input.shape}")
print(f"  Compressed metacell dataset size: {adata_metacells.shape}")

# Export compressed biological matrices for downstream CRISPR perturbation screening
os.makedirs("scanpy_data", exist_ok=True)
output_path = "scanpy_data/adata_metacells.h5ad"
adata_metacells.write_h5ad(output_path)
print(f"Compressed archive saved successfully at: '{output_path}'. Ready for scGPT and Geneformer frameworks.")


## scGPT

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
import scanpy as sc
import torch.nn.functional as F
from scgpt.model import TransformerGenerator

# ==============================================================================
# 1. PIPELINE CONFIGURATION & RUNTIME SETUP
# ==============================================================================
DATA_PATH = "scanpy_data/adata_metacells.h5ad"
BATCH_SIZE = 64  # Size of gene batches for parallel inference on CPU
N_BINS = 51

# Configure target clusters to match your aggregated metacell metadata
GROUP_RESPONDERS = "response"      # Target therapeutic state
GROUP_NON_RESPONDERS = "non-response"  # Baseline state (chemoresistant)
TARGET_COL = "sample2"             # Target observation column in metadata (.obs)

print("Step 1: Loading compressed metacell archive...")
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Compressed file {DATA_PATH} not found. Verify upstream preprocessing paths.")

adata_meta = sc.read_h5ad(DATA_PATH)

# Enforce strict determinism for reproducible production server calculations
torch.manual_seed(42)
np.random.seed(42)

# ==============================================================================
# 2. scGPT INITIALIZATION (PURE PYTHON VOCABULARY ENGINE BYPASS)
# ==============================================================================
print("\nStep 2: Initializing scGPT transformer architecture and checkpoint extraction...")
MODEL_DIR = "/data/docker_data/shared_storage/Grigory/Colorectal/scGPT_pancancer/scGPT_pancancer/"
WEIGHTS_PATH = os.path.join(MODEL_DIR, "best_model.pt")
VOCAB_PATH = os.path.join(MODEL_DIR, "vocab.json")
ARGS_PATH = os.path.join(MODEL_DIR, "args.json")

# 2.1. Dynamic configuration ingestion from hyperparameters manifest
if os.path.exists(ARGS_PATH):
    with open(ARGS_PATH, "r") as f:
        model_args = json.load(f)
    print("  📑 Model execution arguments parsed successfully from args.json.")

    d_model = model_args.get("embsize", model_args.get("d_model", 512))
    nhead = model_args.get("nheads", model_args.get("nhead", 8))
    d_hid = model_args.get("d_hid", 512)
    nlayers = model_args.get("nlayers", 12)
else:
    print(" Configuration file args.json absent. Falling back to default transformer parameters (512/8/12).")
    d_model, nhead, d_hid, nlayers = 512, 8, 512, 12

# 2.2. Native Python Vocabulary Engine compilation
# Workaround: Bypasses native torchtext compilation dependencies to prevent execution panics in Python 3.11+.
print("  🧬 Compiling genomic vocabulary indices directly from serialization manifest...")
with open(VOCAB_PATH, "r") as f:
    raw_vocab_dict = json.load(f)

pad_token = "<pad>"
special_tokens = {pad_token: 0, "<cls>": 1, "<eoc>": 2}

# Guarantee insertion of systemic special structural tokens
for token, idx in special_tokens.items():
    if token not in raw_vocab_dict:
        raw_vocab_dict[token] = len(raw_vocab_dict)

class PurePythonVocab:
    def __init__(self, token_dict, pad_t="<pad>"):
        self.token2idx = token_dict
        self.pad_idx = token_dict[pad_t]

    def __getitem__(self, token):
        # Unmapped panel genes are routed to structural padding to secure runtime stability
        return self.token2idx.get(token, self.pad_idx)

    def __contains__(self, token):
        return token in self.token2idx

    def __len__(self):
        return len(self.token2idx)

    def set_default_index(self, idx):
        pass  # Standard library abstract method structural override

vocab = PurePythonVocab(raw_vocab_dict, pad_t=pad_token)
print(f" Standalone Python vocabulary initialized. Size: {len(vocab)} tokens.")

# 2.3. Instantiating matching pan-cancer foundation transformer instance
model = TransformerGenerator(
    ntoken=len(vocab),
    d_model=d_model,
    nhead=nhead,
    d_hid=d_hid,
    nlayers=nlayers,
    nlayers_cls=1,
    n_cls=1,
    vocab=vocab,
    n_input_bins=N_BINS,
    pad_token=pad_token
)

if os.path.exists(WEIGHTS_PATH):
    print(f"  -> Loading pretrained pan-cancer weight parameters...")
    checkpoint = torch.load(WEIGHTS_PATH, map_location="cpu")
    state_dict = checkpoint.get("model_state_dict", checkpoint)
    model.load_state_dict(state_dict, strict=False)
    print("Pretrained pan-cancer structural state dict mapped successfully!")
else:
    print(" Target checkpoint missing. Model instantiated in test mode using random parameter weights.")

model.to("cpu")
model.eval()

# ==============================================================================
# 3. THERAPEUTIC REVERSAL TRAJECTORY COMPUTATION
# ==============================================================================
print("\n⏳ Step 3: Mapping clinical state latent coordinates using cell-count weights...")

# Extract valid genomic identities filtered against the scGPT vocabulary footprint
gene_ids = np.array([vocab[gene] if gene in vocab else vocab[pad_token] for gene in adata_meta.var_names])
valid_gene_mask = gene_ids != vocab[pad_token]

genes_in_panel = [gene for gene, is_valid in zip(adata_meta.var_names, valid_gene_mask) if is_valid]
expr_filtered = adata_meta.X.toarray()[:, valid_gene_mask] if hasattr(adata_meta.X, "toarray") else adata_meta.X[:, valid_gene_mask]
gene_ids_filtered = gene_ids[valid_gene_mask]

print("  -> Preparing high-precision expression tensors (Float32)...")
values_tensor = torch.tensor(expr_filtered, dtype=torch.float32)
gene_ids_tensor = torch.tensor(gene_ids_filtered, dtype=torch.long).unsqueeze(0).repeat(values_tensor.shape[0], 1)

with torch.no_grad():
    padding_mask_bool = torch.zeros_like(gene_ids_tensor, dtype=torch.bool)
    pert_flags_base = torch.zeros_like(gene_ids_tensor, dtype=torch.long)

    # WORKAROUND: Invoke internal low-level method `_encode` to avoid automatic type casting inside the standard forward call
    print("  -> Extracting baseline embeddings via explicit model._encode()...")
    transformer_output = model._encode(
        src=gene_ids_tensor,
        values=values_tensor,
        input_pert_flags=pert_flags_base,
        src_key_padding_mask=padding_mask_bool
    )

    # Execute mean-pooling over the gene axis (dim 1) to derive consolidated metacell states
    if len(transformer_output.shape) == 3:
        cell_embeddings = transformer_output.mean(dim=1)
    else:
        cell_embeddings = transformer_output

    cell_embeddings = cell_embeddings.cpu().numpy()
    adata_meta.obsm['X_scgpt'] = cell_embeddings

# Clinical response cohort masking validation
if TARGET_COL not in adata_meta.obs.columns:
    raise KeyError(f"Target annotation column '{TARGET_COL}' not detected in metadata matrices!")

responders_mask = adata_meta.obs[TARGET_COL] == GROUP_RESPONDERS
non_responders_mask = adata_meta.obs[TARGET_COL] == GROUP_NON_RESPONDERS

if not responders_mask.any() or not non_responders_mask.any():
    raise ValueError(f"Cohort states '{GROUP_RESPONDERS}' or '{GROUP_NON_RESPONDERS}' missing from categorical variables!")

# Extract metacell scale dimensions (cell_count) for mathematically unbiasing latent centroids
weights_resp = torch.tensor(adata_meta.obs.loc[responders_mask, 'cell_count'].values, dtype=torch.float32)
weights_non_resp = torch.tensor(adata_meta.obs.loc[non_responders_mask, 'cell_count'].values, dtype=torch.float32)

emb_resp = torch.tensor(adata_meta.obsm['X_scgpt'][responders_mask])
emb_non_resp = torch.tensor(adata_meta.obsm['X_scgpt'][non_responders_mask])

# Calculate true weighted structural centroids across clinical groups
E_responders = torch.sum(emb_resp * weights_resp.unsqueeze(1), dim=0) / torch.sum(weights_resp)
E_non_responders = torch.sum(emb_non_resp * weights_non_resp.unsqueeze(1), dim=0) / torch.sum(weights_non_resp)

# Formulate the target reversal vector pointing toward therapeutic sensitivity
V_reversal = E_responders - E_non_responders

print(f"  Therapeutic Centroid mapped using {int(torch.sum(weights_resp))} aggregate single-cell arrays.")
print(f"  Chemoresistant Centroid mapped using {int(torch.sum(weights_non_resp))} aggregate single-cell arrays.")
print(f" Directional therapeutic trajectory [{GROUP_NON_RESPONDERS} -> {GROUP_RESPONDERS}] computed successfully.")

# ==============================================================================
# 4. BATCH VIRTUAL CRISPR PERTURBATION SCREENING
# ==============================================================================
print(f"\n Step 4: Executing parallel in silico knockout screen across {len(genes_in_panel)} targets...")

# Target the most representative, high-density chemoresistant metacell profile
non_resp_indices = np.where(non_responders_mask)[0]
best_non_resp_idx = non_resp_indices[adata_meta.obs.iloc[non_resp_indices]['cell_count'].argmax()]

# Isolate baseline properties for the representative chemoresistant profile
target_expr = values_tensor[best_non_resp_idx].clone()
target_genes = gene_ids_tensor[best_non_resp_idx].clone()
baseline_emb = torch.tensor(adata_meta.obsm['X_scgpt'][best_non_resp_idx])

screening_results = []

# Process the screened genomic panel using vectorized batches
for i in range(0, len(genes_in_panel), BATCH_SIZE):
    batch_genes = genes_in_panel[i:i + BATCH_SIZE]
    current_batch_size = len(batch_genes)

    # Build batch dimension tensors duplicating the target cell template profile
    batch_expr_tensor = target_expr.unsqueeze(0).repeat(current_batch_size, 1)
    batch_gene_tensor = target_genes.unsqueeze(0).repeat(current_batch_size, 1)

    # Initialize basic binary mask patterns and dynamic perturbation arrays
    batch_padding_mask = torch.zeros_like(batch_gene_tensor, dtype=torch.bool)
    batch_pert_flags = torch.zeros_like(batch_gene_tensor, dtype=torch.long)

    # Simulate parallel genetic ablation (CRISPR KO) across current panel slice
    for local_idx, gene_name in enumerate(batch_genes):
        # Identify the global matrix index position of the specific gene within your panel
        global_gene_pos = genes_in_panel.index(gene_name)

        # Enforce expression silencing, insert token masking, and raise perturbation flags
        batch_expr_tensor[local_idx, global_gene_pos] = 0.0
        batch_gene_tensor[local_idx, global_gene_pos] = vocab[pad_token]
        batch_pert_flags[local_idx, global_gene_pos] = 1  # 1 denotes an active knockout state

    with torch.no_grad():
        # High-performance batch evaluation bypassing standard forward method overrides
        perturbed_outputs = model._encode(
            src=batch_gene_tensor,
            values=batch_expr_tensor,
            input_pert_flags=batch_pert_flags,
            src_key_padding_mask=batch_padding_mask
        )

        # Perform downstream pooling across the gene token axis if necessary
        if len(perturbed_outputs.shape) == 3:
            perturbed_embeddings = perturbed_outputs.mean(dim=1)
        else:
            perturbed_embeddings = perturbed_outputs

    # Calculate state shift property displacements induced by virtual intervention
    for local_idx, gene_name in enumerate(batch_genes):
        V_displacement = perturbed_embeddings[local_idx] - baseline_emb

        # Quantify target alignment via cosine similarity metrics against therapeutic trajectory
        cosine_score = F.cosine_similarity(V_displacement.unsqueeze(0), V_reversal.unsqueeze(0)).item()
        screening_results.append({"gene": gene_name, "scgpt_reversal_score": cosine_score})

# ==============================================================================
# 5. RESULTS SERIALIZATION
# ==============================================================================
print("\nStep 5: Formatting and serializing phenotypic transition scores...")

# Consolidate candidate screening metrics into an analytical DataFrame structure
df_scgpt_output = pd.DataFrame(screening_results)
df_scgpt_output = df_scgpt_output.sort_values(by="scgpt_reversal_score", ascending=False).reset_index(drop=True)
df_scgpt_output['rank'] = df_scgpt_output.index + 1

# Export results matrix to storage
os.makedirs("screening_outputs", exist_ok=True)
csv_out_path = "screening_outputs/scgpt_perturbation_leads.csv"
df_scgpt_output.to_csv(csv_out_path, index=False)

print(f" Virtual CRISPR screening complete! Metrics exported to: '{csv_out_path}'")
print("\nTop 5 Reversal Leads Predicted by scGPT:")
print(df_scgpt_output.head(5).to_string(index=False))


## Geneformer

In [ ]:
import os
import pickle
import torch
import numpy as np
import pandas as pd
import scanpy as sc
from transformers import BertModel

# ==============================================================================
# 1. PIPELINE CONFIGURATION & DATA LOADING
# ==============================================================================
BASE_DIR = "/data/docker_data/shared_storage/Grigory/Colorectal"
DATA_PATH = os.path.join(BASE_DIR, "scanpy_data", "adata_metacells.h5ad")

BATCH_SIZE = 64  # Size of gene batches for parallel in silico evaluations
N_BINS = 51

# Configure cohort identifiers to match your compressed metacell parameters
GROUP_RESPONDERS = "response"      # Target therapeutic state
GROUP_NON_RESPONDERS = "non-response"  # Baseline state (chemoresistant)
TARGET_COL = "sample2"             # Target observation column in metadata (.obs)

print(" Step 1: Loading compressed metacell archive...")
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Compressed archive file {DATA_PATH} not found!")

adata = sc.read_h5ad(DATA_PATH)
genes = adata.var_names.tolist()
print(f"  Successfully loaded metacells: {adata.shape} | Total genes in current panel: {len(genes)}")

# ==============================================================================
# 2. OFFLINE ENSEMBL VOCABULARY INGESTION
# ==============================================================================
print("\n Step 2: Extracting structural genomic dictionary from local storage...")

# Hardcoded repository mapping paths for the 316M parameter Geneformer checkpoint
WEIGHTS_DIR = "/data/docker_data/shared_storage/Grigory/Lymphoma/Geneformer/Geneformer-V2-316M"
vocab_path = "/data/docker_data/shared_storage/Grigory/Lymphoma/Geneformer/geneformer/gene_name_id_dict_gc104M.pkl"

if not os.path.exists(vocab_path):
    raise FileNotFoundError(f"Geneformer vocabulary file not found at path: {vocab_path}")

print(f"  🎯 Target Geneformer dictionary verified at: {vocab_path}")

with open(vocab_path, "rb") as f:
    gene_id_dict = pickle.load(f)
print(" Genomic dictionary parsed successfully via direct serialization import.")

# ==============================================================================
# 3. SECURE TOKEN MAPPING & FOUNDATION MODEL INSTANTIATION
# ==============================================================================
print("\n⏳ Step 3: Mapping candidate gene panel onto native Geneformer tokens...")

# Initialize vocabulary structure with native Geneformer special structural tokens
vocab = {"<pad>": 0, "<cls>": 1, "<mask>": 2, "<eoc>": 3}
missing_genes = []

# Index panel genes against the original dictionary keys using strict uppercase alignment
for gene in genes:
    gene_upper = str(gene).upper()
    if gene_upper in gene_id_dict:
        vocab[gene_upper] = gene_id_dict[gene_upper]
    else:
        missing_genes.append(gene)

# Isolate valid panel genes excluding systemic token structures
valid_genes_symbols = [g for g in vocab.keys() if g not in ["<pad>", "<cls>", "<mask>", "<eoc>"]]

print(f" Successfully mapped genes: {len(valid_genes_symbols)} out of {len(genes)}")
if missing_genes:
    print(f"  Warning: {len(missing_genes)} panel genes absent from native Geneformer vocabulary indices.")
    print(f"  Examples of unmapped elements: {missing_genes[:5]}")

print(" Instantiating native architecture layout (BertModel from Hugging Face)...")
model = BertModel.from_pretrained(WEIGHTS_DIR)

model.to("cpu")
model.eval()
print("  Real-world pretrained Geneformer biological weights integrated successfully!")


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
import pickle
from transformers import BertModel

# Load model configuration and parameter weights to determine the exact embedding dimension boundary
print(" Step 2.5: Initializing Geneformer architecture structural properties...")
WEIGHTS_DIR = "/data/docker_data/shared_storage/Grigory/Lymphoma/Geneformer/Geneformer-V2-316M"
model = BertModel.from_pretrained(WEIGHTS_DIR)
model.to("cpu")
model.eval()

# Native vocabulary dimension size (standard default baseline: 25489)
MAX_ALLOWED_TOKEN = model.config.vocab_size
print(f" Model loaded successfully. Maximum valid token ID boundary: {MAX_ALLOWED_TOKEN}")

# ==============================================================================
# 3. MULTI-STAGE GENOMIC MAPPING WITH BOUNDARY PROTECTION
# ==============================================================================
print("\n⏳ Step 3: Executing two-stage panel mapping to numeric token sequences...")

symbol_to_ensembl_path = "/data/docker_data/shared_storage/Grigory/Lymphoma/Geneformer/geneformer/gene_name_id_dict_gc104M.pkl"
with open(symbol_to_ensembl_path, "rb") as f:
    symbol_to_ensembl = pickle.load(f)

ensembl_to_token_path = "/data/docker_data/shared_storage/Grigory/Lymphoma/Geneformer/geneformer/gene_dictionaries_30m/token_dictionary_gc30M.pkl"
with open(ensembl_to_token_path, "rb") as f:
    ensembl_to_token = pickle.load(f)

vocab = {"<pad>": 0, "<cls>": 1, "<mask>": 2, "<eoc>": 3}
missing_genes = []
out_of_range_count = 0

# Map HGNC symbols through Ensembl IDs into filtered Geneformer vocabulary tokens
for gene in genes:
    gene_upper = str(gene).upper()
    if gene_upper in symbol_to_ensembl:
        ensembl_id = symbol_to_ensembl[gene_upper]
        if ensembl_id in ensembl_to_token:
            token_val = int(ensembl_to_token[ensembl_id])

            # BOUNDARY PROTECTION: Verify token matches the instantiated embedding matrix layout
            if token_val < MAX_ALLOWED_TOKEN:
                vocab[gene_upper] = token_val
            else:
                out_of_range_count += 1
                missing_genes.append(gene)
        else:
            missing_genes.append(gene)
    else:
        missing_genes.append(gene)

valid_genes_symbols = [g for g in vocab.keys() if not g.startswith("<")]
print(f" Panel genes successfully linked to valid tokens: {len(valid_genes_symbols)} out of {len(genes)}")
if out_of_range_count > 0:
    print(f"   Filtered out {out_of_range_count} tokens exceeding model vocabulary size limit of {MAX_ALLOWED_TOKEN}.")

# ==============================================================================
# 4. EXPRESSION SORTING & RANK-VALUE ENCODING
# ==============================================================================
print("\n Step 4: Computing non-zero co-expression rank-sorting for metacells...")
raw_expr = adata.X.toarray() if hasattr(adata.X, "toarray") else np.array(adata.X)

MAX_LEN = 2048
gene_ids_list = []

for cell_profile in raw_expr:
    # Sort gene indices based on continuous expression magnitudes in descending order
    sorted_gene_indices = np.argsort(cell_profile)[::-1]
    cell_tokens = []

    for idx in sorted_gene_indices:
        gene_name = str(genes[idx]).upper()
        if gene_name in vocab and not gene_name.startswith("<"):
            cell_tokens.append(int(vocab[gene_name]))

    # Truncate to accommodate structural token additions within the maximum sequence length boundary
    cell_tokens = cell_tokens[:MAX_LEN - 1]
    cell_tokens_with_cls = [int(vocab["<cls>"])] + cell_tokens
    gene_ids_list.append(cell_tokens_with_cls)

# Synchronize array dimensions using structural padding tokens
max_seq_len = max(len(row) for row in gene_ids_list)
padded_gene_ids = []
pad_token = int(vocab["<pad>"])

for tokens in gene_ids_list:
    padding_needed = max_seq_len - len(tokens)
    padded_seq = [int(t) for t in tokens] + [pad_token] * padding_needed
    padded_gene_ids.append(padded_seq)

gene_ids_all = torch.tensor(padded_gene_ids, dtype=torch.long)
attention_mask = (gene_ids_all != pad_token).long()
print(f"Rank-value token matrix generated successfully. Dimensions: {gene_ids_all.shape}")

# ==============================================================================
# 5. THERAPEUTIC REVERSAL TRAJECTORY COMPUTATION
# ==============================================================================
print("\n Step 5: Computing clinical trajectory vectors via hidden state pooling...")
with torch.no_grad():
    outputs = model(input_ids=gene_ids_all, attention_mask=attention_mask)
    all_cell_embeddings = outputs.last_hidden_state[:, 0, :] # Extract latent vectors via the <cls> token position

# Generate condition masks matching target metadata profiles
early_mask = adata.obs[TARGET_COL].astype(str).str.contains(GROUP_NON_RESPONDERS).values
late_mask = adata.obs[TARGET_COL].astype(str).str.contains(GROUP_RESPONDERS).values

# Extract clinical scaling factors (cell_count) to calculate mathematically unbiased centroids
weights_early = torch.tensor(adata.obs.loc[early_mask, 'cell_count'].values, dtype=torch.float32)
weights_late = torch.tensor(adata.obs.loc[late_mask, 'cell_count'].values, dtype=torch.float32)

emb_early = all_cell_embeddings[early_mask]
emb_late = all_cell_embeddings[late_mask]

# Compute true weighted latent centers of gravity across cohorts
E_early = torch.sum(emb_early * weights_early.unsqueeze(1), dim=0) / torch.sum(weights_early)
E_late = torch.sum(emb_late * weights_late.unsqueeze(1), dim=0) / torch.sum(weights_late)

# Formulate reversal trajectory vector directing from chemoresistance to therapeutic sensitivity
V_reversal = E_late - E_early
print(" Weighted directional therapeutic trajectory vector computed successfully.")

# ==============================================================================
# 6. BATCH VIRTUAL CRISPR DELETION SCREENING
# ==============================================================================
print(f"\n Step 6: Initializing parallel in silico knockout screen across panel batches (Size: {BATCH_SIZE})...")

# Identify the highest-density representative chemoresistant profile as the optimization baseline
early_indices = np.where(early_mask)
best_early_idx = early_indices[0][adata.obs.iloc[early_indices[0]]['cell_count'].argmax()]

target_cell_profile = gene_ids_all[best_early_idx : best_early_idx + 1, :]
target_attention_mask = attention_mask[best_early_idx : best_early_idx + 1, :]

# INTEGRITY FIX: Compute baseline state metrics directly from the specific template profile
with torch.no_grad():
    target_init_outputs = model(input_ids=target_cell_profile, attention_mask=target_attention_mask)
    E_target_init = target_init_outputs.last_hidden_state[:, 0, :][0] # 1D Hidden dimension vector array [hidden_dim]

screening_results = {}
valid_genes_to_screen = [g for g in genes if str(g).upper() in vocab and not str(g).startswith("<")]

# Execute vectorized ablation passes across valid screening targets
for i in range(0, len(valid_genes_to_screen), BATCH_SIZE):
    gene_batch = valid_genes_to_screen[i : i + BATCH_SIZE]
    current_batch_size = len(gene_batch)

    # Clone template cell parameters into parallelized batch execution rows
    batch_src = target_cell_profile.repeat(current_batch_size, 1)
    batch_mask = target_attention_mask.repeat(current_batch_size, 1)

    # Substitute target genomic identities across parallel pathways with specialized <mask.> tokens
    for row_idx, gene_name in enumerate(gene_batch):
        gene_token_id = vocab[str(gene_name).upper()]
        pos = (batch_src[row_idx] == gene_token_id).nonzero(as_tuple=True)[0]
        if len(pos) > 0:
            batch_src[row_idx, pos] = vocab["<mask>"]

    with torch.no_grad():
        perturbed_outputs = model(input_ids=batch_src, attention_mask=batch_mask)
        perturbed_embeddings = perturbed_outputs.last_hidden_state[:, 0, :]

    # Evaluate induced phenotype displacement trajectory shifts
    for row_idx, gene_name in enumerate(gene_batch):
        # INTEGRITY FIX: Vector displacement calculated directly against the authentic initial target template
        V_knockout = perturbed_embeddings[row_idx] - E_target_init

        # Calculate alignment metrics against the target clinical trajectory path via cosine similarity
        score = F.cosine_similarity(V_knockout.unsqueeze(0), V_reversal.unsqueeze(0)).item()
        screening_results[gene_name] = score

print("   In silico parallelized Geneformer deletion screening complete!")

# ==============================================================================
# 7. RESULTS SERIALIZATION
# ==============================================================================
print("\n Step 7: Formatting and exporting therapeutic lead matrices...")

# Sort candidates by shift magnitude performance in descending order
top_hits_gf = sorted(screening_results.items(), key=lambda x: x[1], reverse=True)
df_gf = pd.DataFrame(top_hits_gf, columns=['Gene', 'Geneformer_Score'])

FINAL_DIR = os.path.join(BASE_DIR, "scanpy_data")
os.makedirs(FINAL_DIR, exist_ok=True)
FINAL_GF_PATH = os.path.join(FINAL_DIR, "geneformer_screening_leads.csv")

df_gf.to_csv(FINAL_GF_PATH, index=False)
print(f" Geneformer structural ranking saved at: {FINAL_GF_PATH}")

print("\n Top 10 High-Priority Reversal Targets predicted by Geneformer:")
print(df_gf.head(10).to_string(index=False))


In [ ]:
import pandas as pd
import os

# ==============================================================================
# PIPELINE CONFIGURATION AND DIRECTORY PATHS
# ==============================================================================
BASE_DIR = "/data/docker_data/shared_storage/Grigory/Colorectal/scanpy_data/"

# Define path locations for independent model screening logs
PATH_GENEFORMER = os.path.join(BASE_DIR, "geneformer_screening_leads.csv")
PATH_SCGPT = os.path.join(BASE_DIR, "crispr_screening_leads.csv")

# Number of top candidate candidates extracted per model profile for consensus mapping
TOP_N = 100

print("Step 1: Ingesting target screening output matrices...")
if not os.path.exists(PATH_GENEFORMER):
    raise FileNotFoundError(f"Geneformer output archive missing at destination: {PATH_GENEFORMER}")
if not os.path.exists(PATH_SCGPT):
    raise FileNotFoundError(f"scGPT output archive missing at destination: {PATH_SCGPT}")

df_gf = pd.read_csv(PATH_GENEFORMER)
df_sc = pd.read_csv(PATH_SCGPT)

# Enforce strict descending order sort validation based on trajectory score metrics
# Automatically isolates target metric indexes without requiring static string matching
df_gf = df_gf.sort_values(by=df_gf.columns[1], ascending=False).reset_index(drop=True)
df_sc = df_sc.sort_values(by=df_sc.columns[1], ascending=False).reset_index(drop=True)

# Generate non-redundant string sets of high-priority candidate pools
top_gf_genes = set(df_gf.head(TOP_N)['Gene'].tolist())
top_sc_genes = set(df_sc.head(TOP_N)['Gene'].tolist())

# ==============================================================================
# CROSS-ARCHITECTURE CONSENSUS MATRIX DERIVATION
# ==============================================================================
common_genes = top_gf_genes.intersection(top_sc_genes)

print(f"\nSummary Analytics (Threshold Top-{TOP_N}):")
print(f"  Total distinct Geneformer candidates: {len(top_gf_genes)}")
print(f"  Total distinct scGPT candidates: {len(top_sc_genes)}")
print(f"  Identified overlapping consensus targets: {len(common_genes)}")

if len(common_genes) > 0:
    summary_data = []

    # Map intersecting variables back onto matching continuous score fields
    for gene in common_genes:
        score_gf = df_gf.loc[df_gf['Gene'] == gene, df_gf.columns[1]].values[0]
        score_sc = df_sc.loc[df_sc['Gene'] == gene, df_sc.columns[1]].values[0]

        # Calculate dynamic categorical rank locations across architectures
        rank_gf = df_gf[df_gf['Gene'] == gene].index[0] + 1
        rank_sc = df_sc[df_sc['Gene'] == gene].index[0] + 1

        summary_data.append({
            'Gene': gene,
            'Geneformer_Score': score_gf,
            'scGPT_Score': score_sc,
            'Geneformer_Rank': rank_gf,
            'scGPT_Rank': rank_sc
        })

    df_common = pd.DataFrame(summary_data)

    # Order consensus targets by average rank to optimize for joint structural consensus
    df_common['Avg_Rank'] = (df_common['Geneformer_Rank'] + df_common['scGPT_Rank']) / 2.0
    df_common = df_common.sort_values(by='Avg_Rank').reset_index(drop=True)

    # Serialize cross-validated consensus targets manifest to disk
    OUTPUT_PATH = os.path.join(BASE_DIR, "consensus_therapeutic_targets.csv")
    df_common.to_csv(OUTPUT_PATH, index=False)
    print(f"Consensus signature successfully serialized to: {OUTPUT_PATH}")

    print("\nConsensus Therapeutic Targets (Ordered by Model Convergence Priority):")
    print(df_common.to_string(index=False))
else:
    print(f"\n[Warning] Intersection array returned empty for threshold Top-{TOP_N}. Adjust parameter limits.")
